$\textbf{ Pipline for running the QWEN transformers}$

$\textbf{ 1) Imports}$

In [1]:

import math, gc, os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)


/home/maxmagnusson/project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


$\textbf{ 2) Dataset handling}$

In [ ]:
from transformers import DataCollatorWithPadding

MAX_LEN = 768 

class RedditGuidelineEncoded(Dataset):
    def __init__(self, df, tok, max_len=MAX_LEN):
        self.df  = df.fillna("").reset_index(drop=True)
        self.tok = tok
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        body = r.get("body","")
        rule = r.get("rule","")
        pos1 = r.get("positive_example_1","")
        pos2 = r.get("positive_example_2","")
        neg1 = r.get("negative_example_1","")
        neg2 = r.get("negative_example_2","")

        text = (
            "### Comment\n" + body + "\n\n"
            "### Rule\n" + rule + "\n\n"
            "### Context\n"
            + ("Positive examples:\n" if (pos1 or pos2) else "")
            + (pos1 + "\n" if pos1 else "")
            + (pos2 + "\n" if pos2 else "")
            + ("Negative examples:\n" if (neg1 or neg2) else "")
            + (neg1 + "\n" if neg1 else "")
            + (neg2 if neg2 else "")
        )

        enc = self.tok(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,      
            add_special_tokens=True,
        )
        enc["labels"] = int(r["rule_violation"]) 
        return enc


$\textbf{ 3) Load and build DataLoaders}$

In [3]:
name = "Models/qwen3-transformer" 

train_df = pd.read_csv("./DataFolder/train_masked.csv")
val_df   = pd.read_csv("./DataFolder/val_masked.csv")
test_df = pd.read_csv("./DataFolder/test_masked.csv")

label_counts = train_df["rule_violation"].value_counts().to_dict()
weights = train_df["rule_violation"].map(lambda y: 1.0 / label_counts[y]).values
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

tok = AutoTokenizer.from_pretrained(name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

collator = DataCollatorWithPadding(tok, pad_to_multiple_of=8)

train_loader = DataLoader(
    RedditGuidelineEncoded(train_df, tok),
    batch_size=8, shuffle=False,
    sampler=sampler,              
    collate_fn=collator,
    pin_memory=True, num_workers=4, persistent_workers=True,
)

val_loader = DataLoader(
    RedditGuidelineEncoded(val_df, tok),
    batch_size=8, shuffle=False,
    collate_fn=collator,
    pin_memory=True, num_workers=4, persistent_workers=True,
)

test_loader = DataLoader(
    RedditGuidelineEncoded(test_df, tok),
    batch_size=1, shuffle=False,
    collate_fn=collator,
    pin_memory=True, num_workers=4, persistent_workers=True,
)


$\textbf{ 4) Model with proper classification}$

In [4]:
device = "cuda"

model = AutoModelForSequenceClassification.from_pretrained(
    name,
    num_labels=2,
    problem_type="single_label_classification",
)
model.config.pad_token_id = tok.pad_token_id
model.config.label_smoothing_factor = 0.1

if hasattr(model.config, "hidden_dropout_prob"):
    model.config.hidden_dropout_prob = 0.1
if hasattr(model.config, "attention_probs_dropout_prob"):
    model.config.attention_probs_dropout_prob = 0.1


Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Models/qwen3-transformer and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


$\textbf{5: Unfreeze layers}$

In [5]:
def unfreeze_qwen2_cls(model, k_last_layers):
    for p in model.parameters():
        p.requires_grad = False
        
    for p in model.model.layers[-k_last_layers:].parameters():
        p.requires_grad = True

    head_names = ["score", "classifier"]
    found_head = False
    for name_ in head_names:
        if hasattr(model, name_):
            for p in getattr(model, name_).parameters():
                p.requires_grad = True
            found_head = True
            break
   
    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"Trainable tensors: {len(trainable)}")
    for n in trainable[:20]:
        print("  ", n)

unfreeze_qwen2_cls(model, k_last_layers=2)


Trainable tensors: 23
   model.layers.26.self_attn.q_proj.weight
   model.layers.26.self_attn.k_proj.weight
   model.layers.26.self_attn.v_proj.weight
   model.layers.26.self_attn.o_proj.weight
   model.layers.26.self_attn.q_norm.weight
   model.layers.26.self_attn.k_norm.weight
   model.layers.26.mlp.gate_proj.weight
   model.layers.26.mlp.up_proj.weight
   model.layers.26.mlp.down_proj.weight
   model.layers.26.input_layernorm.weight
   model.layers.26.post_attention_layernorm.weight
   model.layers.27.self_attn.q_proj.weight
   model.layers.27.self_attn.k_proj.weight
   model.layers.27.self_attn.v_proj.weight
   model.layers.27.self_attn.o_proj.weight
   model.layers.27.self_attn.q_norm.weight
   model.layers.27.self_attn.k_norm.weight
   model.layers.27.mlp.gate_proj.weight
   model.layers.27.mlp.up_proj.weight
   model.layers.27.mlp.down_proj.weight


$\textbf{6: Model settings}$

In [ ]:
from torch.cuda.amp import GradScaler
import torch.nn as nn

epochs         = 20                
learning_rate  = 2e-4             
weight_decay   = 0.01
grad_accum     = 4
max_grad_norm  = 1.0

use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

if use_cuda:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

trainable_params = [p for p in model.parameters() if p.requires_grad]

decay_params, nodecay_params = [], []
for n, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if n.endswith("bias") or "LayerNorm.weight" in n or "layernorm.weight" in n:
        nodecay_params.append(p)
    else:
        decay_params.append(p)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params,   "weight_decay": weight_decay},
        {"params": nodecay_params, "weight_decay": 0.0},
    ],
    lr=learning_rate,
    betas=(0.9, 0.95),
)

steps_per_epoch    = math.ceil(len(train_loader) / max(1, grad_accum))
num_training_steps = epochs * steps_per_epoch
num_warmup_steps   = int(0.08 * num_training_steps) 

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = GradScaler(enabled=(torch.cuda.is_available() and amp_dtype == torch.float16))

model.to(device)


/tmp/ipykernel_1529/3811183304.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(torch.cuda.is_available() and amp_dtype == torch.float16))


Qwen3ForSequenceClassification(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151669, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_a

$\textbf{7: EMA weights}$

In [ ]:
class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {
            k: v.detach().clone()
            for k, v in model.state_dict().items()
            if getattr(v, "dtype", None) is not None and v.dtype.is_floating_point
        }
        self.backup = {}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def store(self, model):
        self.backup = {
            k: v.detach().clone()
            for k, v in model.state_dict().items()
            if k in self.shadow
        }

    @torch.no_grad()
    def copy_to(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                v.copy_(self.shadow[k])

class DummyEMA:
    def update(self, *args, **kwargs): pass
    def store(self, *args, **kwargs): pass
    def copy_to(self, *args, **kwargs): pass

$\textbf{8: Training loop}\\ \\
\text{Early stop}$

In [ ]:
best_val = float("inf")
patience = 2
stuck = 0
best_state = None
ema_enabled = True
ema = EMA(model, decay=0.999) if ema_enabled else DummyEMA()

for epoch in range(1, epochs + 1):
    model.train()
    torch.cuda.empty_cache(); gc.collect()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    optimizer_steps = 0

    for step, batch in enumerate(train_loader, start=1):
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=amp_dtype):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss / grad_accum

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            should_step = (step % grad_accum == 0) or (step == len(train_loader))
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                optimizer_steps += 1
                ema.update(model)
        else:
            loss.backward()
            should_step = (step % grad_accum == 0) or (step == len(train_loader))
            if should_step:
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                optimizer_steps += 1
                ema.update(model)

        running_loss += loss.detach().item()

    avg_train_loss = running_loss / max(1, len(train_loader))
    print(f"Epoch {epoch} | steps: {optimizer_steps}/{steps_per_epoch} | train loss: {avg_train_loss:.4f}")

    model.eval()
    ema.store(model)     
    ema.copy_to(model)

    correct = 0
    total   = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels         = batch["labels"].to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=amp_dtype):
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss_sum += out.loss.item()

            preds = out.logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.numel()

    val_acc  = correct / max(1, total)
    val_loss = val_loss_sum / max(1, len(val_loader))
    print(f"Epoch {epoch}: val loss = {val_loss:.4f} | val acc = {val_acc:.4f}")

    
    if val_loss < best_val:
        best_val = val_loss
        stuck = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stuck += 1
        if stuck > patience:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"Loaded best checkpoint with val loss {best_val:.4f}")


## *Evaluating Model Performance with test set*

In [ ]:
correct = 0
total   = 0
model.eval()                    


with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=amp_dtype):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        preds = out.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.numel()

test_acc  = correct / max(1, total)
print(f"Test acc = {test_acc:.4f}")